# Thử tạo cảnh Origami B2B 16:9 bằng Wan 2.2 trên Kaggle

Notebook này tạo một cảnh liền mạch khoảng 10 giây theo brief PAIN: nhập liệu thủ công tốn thời gian và dữ liệu phân tán. Nhân vật, đạo cụ và bối cảnh đều là mô hình giấy origami; bảng màu navy, trắng ngà và teal; không chữ, số, logo, người thật, bàn tay thật hoặc thao tác gấp giấy.

Cảnh được sinh thành bốn đoạn I2V ngắn nối tuần tự bằng khung hình cuối của đoạn trước. Cách này giảm tải VRAM so với một lượt 161 khung và giúp giữ nhân vật/bối cảnh ổn định hơn. Kết quả vẫn cần người xem duyệt về chuyển động, tính liên tục và việc thẻ rơi vào khe có đọc được hay không.

## Trước khi chạy

1. Mở notebook trong Kaggle, bật **GPU** và **Internet** trong Notebook Settings. Tài liệu Kaggle hiện hướng dẫn GPU trong Settings và lưu ý GPU miễn phí có thể phải chờ cấp.
2. Đọc điều khoản Kaggle và giấy phép model/workflow trước khi tải khoảng 16.9 GB model.
3. Trong ô cấu hình, chỉ bật `BAT_DAU_TAI_MODEL` sau khi đã sẵn sàng tải. Notebook không mở tunnel hoặc API công khai.
4. Chạy tuần tự từ đầu đến cuối. Mỗi GPU chỉ chạy một inference tại một thời điểm; không chạy song song.

Tệp kết quả được lưu trong `/kaggle/working/wan_origami_b2b_16x9_10s/`. Khi phiên hoàn tất, tải MP4 và manifest từ phần Output của Kaggle.


In [ ]:
from pathlib import Path
# Chỉ bật tải sau khi bạn đã đọc điều khoản Kaggle và giấy phép model.
DIEU_KHOAN_DA_DOC = False
BAT_DAU_TAI_MODEL = False

# Hồ sơ ngang 16:9. Bốn đoạn 41 khung ở 16 fps dài tổng cộng khoảng 10.25 giây.
PROFILE = {"width": 832, "height": 480, "frames": 41, "fps": 16, "steps": 24, "cfg": 5.0}

# Đường dẫn làm việc tạm chứa ComfyUI và checkpoint; đầu ra được lưu riêng ở /kaggle/working.
WORK_ROOT = Path("/kaggle/temp/wan_origami_b2b")
OUTPUT_ROOT = Path("/kaggle/working/wan_origami_b2b_16x9_10s")
SEGMENT_ROOT = OUTPUT_ROOT / "segments"

# Hành động được chia nhỏ theo nhịp brief, giữ cùng cảnh và hướng máy quay.
SCENES = [
    {
        "name": "doan_01_phieu_va_so",
        "kind": "t2v",
        "prompt": "Cảnh mở đầu của cùng một cú máy cinematic ngang 16:9, nhìn từ trên chếch xuống bàn vận hành B2B. Toàn bộ là mô hình giấy origami hoàn thiện: bàn giấy trắng ngà có thớ mịn và nếp gấp rõ, ánh sáng mềm, bóng đổ giấy nhất quán; bảng màu navy, trắng ngà, teal. Bên trái đặt nhiều phiếu nguồn rời màu ngà, ở giữa là sổ dữ liệu trống, bên phải là các thẻ công việc rời màu teal và navy, sát cạnh phải có khe hẹp tối của bàn. Có một nhân viên origami nhỏ với dáng người hoàn chỉnh, đủ hai tay hai chân, mặt origami cân đối, hai mắt cùng cao độ, mũi giấy nhỏ, miệng khép thân thiện. Nhân viên dùng cánh tay giấy có đầu dạng khối đơn giản không ngón, lần lượt lấy phiếu bên trái đặt cạnh sổ giữa rồi viết bằng bút giấy, chỉ các nét trừu tượng không đọc được. Máy quay bắt đầu trôi thật chậm từ trái sang phải, tốc độ đều. Không cắt cảnh. Không chữ hoặc số đọc được.",
    },
    {
        "name": "doan_02_doi_chieu_the",
        "kind": "i2v",
        "prompt": "Tiếp diễn trực tiếp từ khung hình đầu vào trong cùng một cảnh và cùng cú máy. Giữ nguyên nhân viên origami, gương mặt, tỷ lệ, bàn, ánh sáng, màu navy/trắng ngà/teal, vị trí sổ và hướng máy quay; không tạo thêm vật thể. Nhân viên tiếp tục ghi các nét trừu tượng không đọc được vào sổ rồi dùng hai cánh tay giấy dạng khối không ngón để so từng phiếu với chồng thẻ công việc bên phải. Các thẻ còn lại vẫn tách rời và nhìn rõ. Chuyển động có nguyên nhân, mềm, không đứng chờ. Không chữ hoặc số đọc được.",
    },
    {
        "name": "doan_03_the_roi_xuong_khe",
        "kind": "i2v",
        "prompt": "Tiếp diễn trực tiếp từ khung hình đầu vào, không đổi cảnh. Giữ nguyên đúng nhân viên origami, bàn giấy, sổ, ánh sáng, bảng màu và chuyển động camera chậm từ trái sang phải. Khi nhân viên đang đối chiếu phiếu với thẻ, đúng một thẻ teal ở mép chồng trượt ra do va chạm nhìn thấy được, đi theo trọng lực và rơi vào khe hẹp cạnh bàn; thẻ khuất hẳn dưới mép bàn. Những thẻ khác vẫn ở bên phải. Không biến hình, không nhân đôi, không tự đổi vị trí vật thể. Không chữ hoặc số đọc được.",
    },
    {
        "name": "doan_04_tiep_tuc_nhap_lieu",
        "kind": "i2v",
        "prompt": "Tiếp diễn trực tiếp từ khung hình đầu vào trong cùng cảnh liền mạch. Giữ nguyên nhân viên origami, bàn, sổ, các phiếu và thẻ còn lại, khe cạnh bàn, ánh sáng mềm và bảng màu navy/trắng ngà/teal. Thẻ đã rơi vẫn khuất dưới mép bàn. Nhân viên tiếp tục nhập các phiếu còn lại vào sổ bằng bút giấy, các nhóm dữ liệu rời rạc vẫn nhìn thấy rõ; máy quay tiếp tục trôi chậm cùng hướng rồi dừng êm. Không thêm hành động hoặc nhân vật. Không chữ, số, phụ đề, logo hoặc watermark.",
    },
]
NEGATIVE_PROMPT = "chữ, số, phụ đề, logo, watermark, người thật, da thật, bàn tay thật, ngón tay, thao tác gấp giấy, biến hình, nhân đôi vật thể, đổi chất liệu sang nhựa hoặc ảnh thật, mặt trống, mắt lệch, mắt lác, miệng méo, thiếu chi, cơ thể dị dạng, quái vật, rung giật, nhấp nháy, đổi cảnh, đổi màu, mờ chủ thể, thẻ biến mất không có nguyên nhân"

if not DIEU_KHOAN_DA_DOC:
    raise RuntimeError("Hãy đọc điều khoản Kaggle và giấy phép model rồi mới bật DIEU_KHOAN_DA_DOC.")
if not BAT_DAU_TAI_MODEL:
    raise RuntimeError("Hãy bật BAT_DAU_TAI_MODEL khi bạn sẵn sàng tải khoảng 16.9 GB model.")


## 1. Kiểm tra Kaggle, GPU, dung lượng và mạng

Đoạn này dừng nếu thiếu GPU NVIDIA, dung lượng scratch hoặc Internet tới Hugging Face. Notebook dùng GPU 0 và xử lý tuần tự.


In [ ]:
import os, sys, json, time, shutil, socket, subprocess, hashlib, urllib.request
from pathlib import Path

if not Path("/kaggle").exists() or not Path("/kaggle/temp").exists():
    raise RuntimeError("Hãy mở notebook trong Kaggle Notebooks.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Chưa bật GPU trong Kaggle Notebook Settings.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
gpus = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
if not gpus:
    raise RuntimeError("Không thấy GPU NVIDIA.")
print("GPU được cấp:", gpus)
print("Chỉ dùng GPU 0; concurrency = 1.")
if int(gpus[0].split(",")[-1].strip().split()[0]) < 12000:
    raise RuntimeError("GPU 0 có dưới 12 GiB VRAM; dừng trước khi tải model.")
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("Kaggle image cần có ffmpeg và ffprobe.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SEGMENT_ROOT.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(WORK_ROOT).free / 1024**3
ram_gib = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3, 2)
print({"scratch_con_gib": round(free_gib, 2), "ram_gib": ram_gib})
if free_gib < 22:
    raise RuntimeError("Cần ít nhất 22 GiB scratch trống để tải model và chạy ComfyUI.")
try:
    with urllib.request.urlopen("https://huggingface.co/api/models/Comfy-Org/Wan_2.2_ComfyUI_Repackaged", timeout=20) as response:
        print("Hugging Face HTTPS:", response.status)
except Exception as exc:
    raise RuntimeError("Bật Internet trong Kaggle Settings rồi chạy lại: " + str(exc))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


## 2. Cài ComfyUI theo phiên bản đã ghim

ComfyUI và workflow được ghim theo commit/hash. Các gói Python được cài trong runtime Kaggle, không thay đổi notebook chuẩn hoặc repository BAHA.


In [ ]:
COMFY_COMMIT = "b5a432157de6b8884f81104cd02926adf4d597e4"
WORKFLOW_COMMIT = "abb9cf6477b59c52dbe234c67f82318dadd872b2"
WORKFLOW_SHA256 = "23C93B88ADECBDF5BCF4C5438117070FE757F30ACA373DD6A2EB488D2C6A585A"
COMFY = WORK_ROOT / "ComfyUI"
if not COMFY.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
subprocess.run(["git", "-C", str(COMFY), "checkout", "--detach", COMFY_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(COMFY / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "requests"], check=True)
print("ComfyUI đã ghim:", COMFY_COMMIT)


## 3. Tải checkpoint Wan và xác minh SHA-256

Notebook tải ba tệp model công khai theo revision cố định. Tổng dung lượng xấp xỉ 16.9 GB. Nếu tải gián đoạn, chạy lại cell này; Hugging Face Hub tiếp tục dùng cache hiện có.


In [ ]:
import requests
from huggingface_hub import hf_hub_download

MODEL_FILES = [
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "ee6f4a40737a995bf5818954cfce6d59443b0f04", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", COMFY / "models/diffusion_models"),
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "ee6f4a40737a995bf5818954cfce6d59443b0f04", "split_files/vae/wan2.2_vae.safetensors", COMFY / "models/vae"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", COMFY / "models/text_encoders"),
]
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

model_locks = {}
for repo, revision, filename, destination in MODEL_FILES:
    destination.mkdir(parents=True, exist_ok=True)
    start = time.monotonic()
    path = Path(hf_hub_download(repo_id=repo, filename=filename, revision=revision,
                                local_dir=destination, cache_dir=WORK_ROOT / "hf_cache"))
    model_locks[repo + "/" + filename] = {
        "revision": revision, "sha256": sha256_file(path),
        "bytes": path.stat().st_size, "download_seconds": round(time.monotonic() - start, 2)
    }
print(json.dumps(model_locks, ensure_ascii=False, indent=2))


## 4. Khởi động ComfyUI nội bộ và nạp workflow

ComfyUI chỉ lắng nghe loopback trong phiên Kaggle. Không có cloudflared, ngrok, tunnel hoặc API công khai.


In [ ]:
COMFY_URL = "http://127.0.0.1:8188"
server_log = open(WORK_ROOT / "comfyui.log", "w", encoding="utf-8")
server = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                         cwd=COMFY, stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI dừng khi khởi động; xem " + str(WORK_ROOT / "comfyui.log"))
    try:
        if requests.get(COMFY_URL + "/system_stats", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError("ComfyUI chưa sẵn sàng sau 4 phút.")
object_info = requests.get(COMFY_URL + "/object_info", timeout=30).json()
raw_workflow = requests.get(
    f"https://raw.githubusercontent.com/Comfy-Org/workflow_templates/{WORKFLOW_COMMIT}/templates/video_wan2_2_5B_ti2v.json",
    timeout=30)
raw_workflow.raise_for_status()
if hashlib.sha256(raw_workflow.content).hexdigest().upper() != WORKFLOW_SHA256:
    raise RuntimeError("SHA-256 workflow không khớp; dừng để rà soát nguồn.")
template = raw_workflow.json()
template["nodes"] = [node for node in template["nodes"] if node.get("type") != "MarkdownNote"]
print("ComfyUI sẵn sàng; workflow hash khớp.")


## 5. Chuẩn bị API graph và cơ chế chạy tuần tự

Mỗi đoạn chỉ gửi một job. Khi ComfyUI từ chối, dừng với lỗi; khi trạng thái không rõ, notebook không tự gửi lại để tránh tạo inference trùng.


In [ ]:
import copy

WORKFLOW_URL = f"https://raw.githubusercontent.com/Comfy-Org/workflow_templates/{WORKFLOW_COMMIT}/templates/video_wan2_2_5B_ti2v.json"
STATE_PATH = WORK_ROOT / "trang_thai_run.json"
run_state = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}

def save_state():
    STATE_PATH.write_text(json.dumps(run_state, ensure_ascii=False, indent=2), encoding="utf-8")

def widget(workflow, node_id, index, value):
    next(node for node in workflow["nodes"] if node["id"] == node_id)["widgets_values"][index] = value

def workflow_to_api(workflow):
    links = {item[0]: item[1:] for item in workflow.get("links", [])}
    active = {node["id"] for node in workflow["nodes"] if node.get("mode", 0) not in (2, 4)}
    graph = {}
    for node in workflow["nodes"]:
        if node["id"] not in active:
            continue
        node_type = node["type"]
        info = object_info.get(node_type)
        if not info:
            raise RuntimeError("ComfyUI thiếu node: " + node_type)
        raw_definitions = info.get("input", {})
        definitions = {**raw_definitions.get("required", {}), **raw_definitions.get("optional", {})}
        sockets = {item["name"] for item in node.get("inputs", [])}
        names = [key for group in ("required", "optional") for key in raw_definitions.get(group, {}) if key not in sockets]
        values = list(node.get("widgets_values", []))
        if node_type == "KSampler" and len(values) == 7:
            values.pop(1)
        if node_type == "LoadImage":
            names, values = ["image"], values[:1]
        if len(values) < len(names):
            for missing in names[len(values):]:
                definition = definitions.get(missing, [None, {}])
                config = definition[1] if len(definition) > 1 else {}
                default = config.get("default")
                if default is None and config.get("options"):
                    default = config["options"][0]
                choices = definition[0] if definition and isinstance(definition[0], list) else None
                if default is None and choices:
                    default = choices[0][0] if len(choices) == 1 and isinstance(choices[0], list) else choices[0]
                values.append(default)
        if len(names) != len(values):
            raise RuntimeError(f"Không ánh xạ được widget node {node_type}: {names} / {values}")
        inputs = dict(zip(names, values))
        for key, value in list(inputs.items()):
            definition = definitions.get(key, [None, {}])
            choices = definition[0] if definition and isinstance(definition[0], list) else None
            if choices and len(choices) == 1 and isinstance(choices[0], list):
                choices = choices[0]
            if choices and value not in choices and isinstance(value, str):
                matches = [choice for choice in choices if isinstance(choice, str) and choice.endswith("/" + value)]
                if len(matches) == 1:
                    inputs[key] = matches[0]
        for item in node.get("inputs", []):
            link_id = item.get("link")
            if link_id is None or link_id not in links:
                continue
            origin, slot, target_id, _, _ = links[link_id]
            if origin in active and target_id == node["id"]:
                inputs[item["name"]] = [str(origin), int(slot)]
        graph[str(node["id"])] = {"class_type": node_type, "inputs": inputs,
                                  "_meta": {"title": node.get("title", node_type)}}
    return graph

def prepare_workflow(name, prompt, seed, image_name=None):
    workflow = copy.deepcopy(template)
    widget(workflow, 55, 0, PROFILE["width"])
    widget(workflow, 55, 1, PROFILE["height"])
    widget(workflow, 55, 2, PROFILE["frames"])
    widget(workflow, 57, 0, PROFILE["fps"])
    widget(workflow, 58, 0, "video/wan_" + name)
    widget(workflow, 6, 0, prompt)
    widget(workflow, 7, 0, NEGATIVE_PROMPT)
    widget(workflow, 3, 0, int(seed))
    widget(workflow, 3, 1, "fixed")
    widget(workflow, 3, 2, PROFILE["steps"])
    widget(workflow, 3, 3, PROFILE["cfg"])
    image_node = next(node for node in workflow["nodes"] if node["id"] == 56)
    if image_name:
        image_node["mode"] = 0
        widget(workflow, 56, 0, image_name)
    else:
        image_node["mode"] = 4
    return workflow_to_api(workflow)

# Kiểm tra hợp đồng graph mà không gửi inference.
probe = prepare_workflow("kiem_tra", SCENES[0]["prompt"], 90101)
assert probe["55"]["inputs"]["width"] == PROFILE["width"]
assert probe["55"]["inputs"]["height"] == PROFILE["height"]
assert probe["55"]["inputs"]["length"] == PROFILE["frames"]
assert "start_image" not in probe["55"]["inputs"]
print("Graph T2V hợp lệ theo kích thước ngang.")


In [ ]:
def submit_segment(scene, seed, image_name=None):
    name = scene["name"]
    old = run_state.get(name)
    if old:
        if old.get("status") == "completed" and Path(old.get("path", "")).exists():
            return Path(old["path"]), old
        raise RuntimeError(f"Đoạn {name} có trạng thái {old.get('status')}; không tự gửi lại. Hãy đối soát queue/history trước.")
    graph = prepare_workflow(name, scene["prompt"], seed, image_name)
    run_state[name] = {"status": "submitting", "seed": seed,
                       "prompt_sha256": hashlib.sha256(scene["prompt"].encode()).hexdigest(),
                       "started_at": time.time()}
    save_state()
    try:
        response = requests.post(COMFY_URL + "/prompt", json={"prompt": graph}, timeout=(5, 60))
        if response.status_code >= 400:
            raise RuntimeError("ComfyUI từ chối job: " + response.text[:2000])
        prompt_id = response.json()["prompt_id"]
    except Exception:
        run_state[name]["status"] = "unknown"
        save_state()
        raise
    run_state[name].update(status="queued", prompt_id=prompt_id)
    save_state()
    job_started = time.monotonic()
    deadline = job_started + 7200
    while time.monotonic() < deadline:
        history = requests.get(COMFY_URL + "/history/" + prompt_id, timeout=20).json()
        entry = history.get(prompt_id)
        if entry and entry.get("outputs"):
            videos = entry["outputs"].get("58", {}).get("videos", [])
            if not videos:
                candidates = sorted((COMFY / "output" / "video").glob(f"wan_{name}_*.mp4"))
                videos = [{"subfolder": "video", "filename": path.name} for path in candidates]
            if videos:
                item = videos[0]
                output = (COMFY / "output" / item.get("subfolder", "") / item["filename"]).resolve()
                if COMFY / "output" not in output.parents:
                    raise RuntimeError("Đường dẫn output nằm ngoài ComfyUI/output.")
                run_state[name].update(status="completed", path=str(output),
                                       bytes=output.stat().st_size,
                                       elapsed_seconds=round(time.monotonic() - job_started, 2))
                save_state()
                return output, run_state[name]
        if server.poll() is not None:
            run_state[name]["status"] = "unknown"
            save_state()
            raise RuntimeError("ComfyUI dừng giữa inference; không tự gửi lại.")
        time.sleep(3)
    run_state[name]["status"] = "unknown"
    save_state()
    raise TimeoutError("Hết thời gian chờ; trạng thái được giữ để tránh inference trùng.")

def extract_last_frame(video, destination):
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-sseof", "-0.08", "-i", str(video), "-frames:v", "1", str(destination)], check=True)
    return Path(destination)

completed_segments = []
for index, scene in enumerate(SCENES):
    persisted_raw = SEGMENT_ROOT / f"{scene['name']}_goc.mp4"
    if persisted_raw.exists():
        source = persisted_raw
        info = {"status": "completed", "path": str(source), "resumed_from_output": True}
    else:
        image_name = None
        if index > 0:
            previous = completed_segments[-1]
            frame = extract_last_frame(previous, WORK_ROOT / f"khung_noi_tiep_{index:02d}.png")
            image_name = frame.name
            shutil.copy2(frame, COMFY / "input" / image_name)
        source, info = submit_segment(scene, 91000 + index, image_name)
        shutil.copy2(source, persisted_raw)
    completed_segments.append(persisted_raw)
    print(scene["name"], info)


## 6. Ghép một cảnh ngang 10 giây và lưu manifest

Bốn đoạn được nối đúng thứ tự; mỗi đoạn bắt đầu từ khung cuối đoạn trước. FFmpeg chuẩn hóa video sang 1920×1080/30 fps và cắt tổng thời lượng còn 10 giây. Không thêm chữ, phụ đề, logo, watermark hoặc âm thanh.


In [ ]:
def probe_video(path):
    data = json.loads(subprocess.check_output([
        "ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)
    ], text=True))
    video = next(stream for stream in data["streams"] if stream["codec_type"] == "video")
    return {"width": video["width"], "height": video["height"],
            "fps": video["avg_frame_rate"], "duration_seconds": float(data["format"]["duration"]),
            "codec": video["codec_name"], "bytes": Path(path).stat().st_size}

concat_list = WORK_ROOT / "danh_sach_ghep.txt"
concat_list.write_text("".join("file '" + str(path.resolve()) + "'\n" for path in completed_segments), encoding="utf-8")
FINAL_VIDEO = OUTPUT_ROOT / "video_origami_b2b_16x9_10s.mp4"
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "concat", "-safe", "0",
    "-i", str(concat_list), "-vf",
    "scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,fps=30",
    "-t", "10", "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
    str(FINAL_VIDEO)
], check=True)

manifest = {
    "platform": "Kaggle",
    "benchmark_only": True,
    "tunnel_opened": False,
    "model": "Wan2.2-TI2V-5B",
    "comfyui_commit": COMFY_COMMIT,
    "workflow_commit": WORKFLOW_COMMIT,
    "workflow_sha256": WORKFLOW_SHA256,
    "profile": PROFILE,
    "gpu": gpus,
    "ram_gib": ram_gib,
    "model_locks": model_locks,
    "segments": [
        {"name": scene["name"], "path": str(path), "sha256": sha256_file(path),
         "bytes": path.stat().st_size}
        for scene, path in zip(SCENES, completed_segments)
    ],
    "final_video": {"path": str(FINAL_VIDEO), "sha256": sha256_file(FINAL_VIDEO),
                    "probe": probe_video(FINAL_VIDEO)},
    "run_state": run_state,
}
manifest_path = OUTPUT_ROOT / "manifest_wan_origami_b2b.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest["final_video"], ensure_ascii=False, indent=2))
print("Tệp đã lưu:", OUTPUT_ROOT)


## 7. Dừng ComfyUI

Chạy sau khi đã kiểm tra MP4 và tải các tệp cần giữ từ Kaggle Output.


In [ ]:
if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server.kill()
print("Đã dừng ComfyUI nội bộ.")
